# Colab training driver

Thin front-end for the package: the first cell bootstraps (mount Drive, read config, clone, install); the rest are calls into `src.colab` and the experiment drivers.

**Configure a run** by creating a config file on your Drive at `MyDrive/cross-dataset-drift-soybean-disease.env` — copy the repo's `.env.example` and fill it in: `GITHUB_PAT` (a read-only token), `GITHUB_REPO_URL`, `GIT_BRANCH`, `DRIVE_ROOT`. The bootstrap reads that file (then environment variables, then Colab Secrets), so nothing account-specific is baked into the notebook and anyone can re-run by editing their own config file.

Dataset zips live at `MyDrive/<DRIVE_ROOT>/data/raw/{ASDID,MH-SoyaHealthVision,PlantVillage}.zip`; checkpoints/results/logs persist under `MyDrive/<DRIVE_ROOT>/`. Deps install from `requirements.txt`, so this runs on whatever Python Colab provides (local dev is pinned to 3.12).

**Which experiment runs is decided by the config's `experiment:` key** (see the run cell), so swapping the `CONFIG` filename actually changes what executes.

## 1 · Setup — run every session
Mount Drive, clone the repo at your branch, install dependencies. Re-run cells 1 and 2 top-to-bottom in every new Colab session.

In [6]:
# --- Bootstrap: mount Drive, read config, clone, install (Colab) ---
import os, shutil, subprocess, sys
from pathlib import Path

# Reduce CUDA fragmentation OOMs on the long runs (set before torch is imported anywhere).
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

from google.colab import drive
drive.mount('/content/drive')

# Edit this file on your Drive to configure a run (copy from the repo's .env.example).
CONFIG_ENV = Path('/content/drive/MyDrive/cross-dataset-drift-soybean-disease.env')
KEYS = ('GITHUB_PAT', 'GITHUB_REPO_URL', 'GIT_BRANCH', 'DRIVE_ROOT')

def _parse_env(path):
    out = {}
    if path.exists():
        for line in path.read_text().splitlines():
            line = line.strip()
            if line and not line.startswith('#') and '=' in line:
                k, v = line.split('=', 1)
                out[k.strip()] = v.strip()
    return out

# 1) Drive config file, 2) environment, 3) Colab Secrets (last; unavailable outside the Colab UI)
SECRETS = _parse_env(CONFIG_ENV)
for k in KEYS:
    if os.environ.get(k):
        SECRETS[k] = os.environ[k]
if not SECRETS.get('GITHUB_PAT'):
    try:    
        from google.colab import userdata
        SECRETS['GITHUB_PAT'] = userdata.get('GITHUB_PAT')
    except Exception as e:
        print('Colab Secrets unavailable:', type(e).__name__)

missing = [k for k in ('GITHUB_PAT', 'GITHUB_REPO_URL', 'DRIVE_ROOT') if not SECRETS.get(k)]
if missing:
    raise RuntimeError(f'Missing {missing}. Create {CONFIG_ENV} from the repo .env.example (or set them as env vars).')

PAT        = SECRETS['GITHUB_PAT']
REPO       = SECRETS['GITHUB_REPO_URL']
BRANCH     = SECRETS.get('GIT_BRANCH', 'main')
DRIVE_ROOT = SECRETS['DRIVE_ROOT']

CODE = Path('/content/code')
if CODE.exists():
    shutil.rmtree(CODE)
url = REPO.replace('https://', f'https://{PAT}@')
subprocess.run(['git', 'clone', '--depth=1', '--branch', BRANCH, url, str(CODE)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(CODE / 'requirements.txt')], check=True)
if str(CODE) not in sys.path:
    sys.path.insert(0, str(CODE))

# Drop any previously imported copy of the package. Re-running this cell replaces
# the files on disk, but Python keeps serving modules it has already imported, so
# without this you silently run the OLD code against the NEW commit. Re-run cells
# 1 -> 2 -> 3 in order after this, so `paths` is rebuilt from the fresh classes.
_stale = [m for m in sys.modules if m.split('.')[0] in ('src', 'scripts')]
for _name in _stale:
    del sys.modules[_name]
print('bootstrap done:', CODE, f'| dropped {len(_stale)} stale module(s)')
print('NOTE: if requirements upgraded torch, restart the runtime before continuing.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
bootstrap done: /content/code


In [7]:
# --- Sanity check: confirm the clone is the version you expect ---
import subprocess
print(subprocess.run(['git', '-C', str(CODE), 'log', '-1', '--oneline'],
                     capture_output=True, text=True).stdout.strip())

# Guard against a stale import: the package that runs must be the one just cloned.
import src
assert Path(src.__file__).resolve().parent.parent == CODE.resolve(), (
    f'imported src from {src.__file__}, not {CODE}. Restart the runtime and re-run.'
)
print('package:', Path(src.__file__).parent)

from src.config import load_config
from src.experiments._common import seed_pairs
cfg = load_config()
print('seed_design:', cfg.seed_design)
print('pairs:', seed_pairs(cfg))

9e3cb20 Compute Confusion Matrices
seed_design: cross
pairs: [(73, 73), (73, 7), (73, 21), (7, 73), (7, 7), (7, 21), (21, 73), (21, 7), (21, 21)]


## 2 · Data & paths — run every session
Cache datasets to local SSD and resolve `paths` (data on SSD, checkpoints/results on Drive). `paths` is consumed by every stage below.

In [8]:
# --- Cache datasets to local SSD, pre-resize once, target artifacts at Drive ---
import src.colab as colab

DRIVE = Path('/content/drive/MyDrive') / DRIVE_ROOT
LOCAL = Path('/content/local')

local_raw = colab.cache_datasets(DRIVE / 'data' / 'raw', LOCAL)
colab.pre_resize_images(local_raw)
paths = colab.resolve_paths(LOCAL, DRIVE)
print('data:', paths.data_root, '| artifacts:', paths.checkpoints_dir)

data: /content/local/data/raw | artifacts: /content/drive/MyDrive/publications/cross-dataset-drift-soybean-disease/checkpoints


## 3 · Run a stage: training **or** evaluation
Dispatch is by the config's `experiment:` key — change `CONFIG` to pick what runs:

- **Training** (writes checkpoints): `full_finetune.yaml`, `unweighted_mh.yaml`, `label_smoothing.yaml`, `control_study.yaml`, `linear_probe.yaml`, `linear_solvability.yaml`
- **Evaluation** (reads checkpoints → `eval_results.csv`): `evaluate.yaml` / `evaluate_resume.yaml`
- **Input-level interventions** (read checkpoints → their own CSVs): `background_intervention.yaml`, `frequency_intervention.yaml`

Run the training configs first (GPU), then evaluation. **Skip this stage** if the checkpoints and `eval_results.csv` already exist on Drive.

The interventions are inference-only and need no retraining. Only `background_intervention.yaml` needs the foreground masks: set `MASKS_DIR` below to wherever they live on Drive (they are gitignored, so the clone never carries them). Leave `MASKS_DIR = None` for every other stage — it is ignored.

In [ ]:
# --- Run an experiment (training only; evaluation runs locally) ---
# Dispatch is by the config's `experiment:` key, so changing CONFIG changes what runs.
#   full_finetune.yaml            -> baseline 16-model space      (checkpoints/finetune)
#   unweighted_mh.yaml            -> unweighted-loss ablation     (checkpoints/finetune_unweighted)
#   label_smoothing.yaml          -> label-smoothing variant      (checkpoints/finetune_label_smoothing)
#   control_study.yaml            -> matched-ASDID control study  (checkpoints/control_study)
#   linear_probe.yaml             -> frozen-feature probes        (checkpoints/linear_probe)
#   calibration.yaml              -> quick single-seed timing pass
#   background_intervention.yaml  -> background replacement       (needs MASKS_DIR)
#   frequency_intervention.yaml   -> frequency low-pass control   (no masks needed)
import importlib, logging
import yaml
logging.basicConfig(level=logging.INFO, format='%(message)s')
from src.config import load_config

from scripts.run_experiment import EXPERIMENTS  # single source of truth (includes linear_solvability)

CONFIG = CODE / 'configs' / 'experiments' / 'evaluate_resume.yaml'   # <-- change this per run

# Foreground masks, for background_intervention only. Not version-controlled, so
# point this at the copy on Drive. None (the default) leaves paths untouched.
MASKS_DIR = None
# MASKS_DIR = '/content/drive/MyDrive/Master_Thesis/Publication/code/data/masks'

experiment = (yaml.safe_load(CONFIG.read_text()) or {}).get('experiment')
assert experiment in EXPERIMENTS, f"{CONFIG.name} must set experiment: to one of {sorted(EXPERIMENTS)}"
run_paths = paths.with_overrides(masks_dir=MASKS_DIR)   # with_overrides ignores None
if experiment == 'background_intervention':
    n_masks = len(list(Path(run_paths.masks_dir).glob('*/*/*_mask.png')))
    print(f'masks: {run_paths.masks_dir}  ({n_masks} found)')
    assert n_masks, f'no masks under {run_paths.masks_dir} -- set MASKS_DIR above'
cfg = load_config(str(CONFIG), paths=run_paths)
# run_name only governs the finetune driver's checkpoint namespace; the
# control_study / linear_probe / evaluate / intervention drivers use their own.
tag = f'  (run_name={cfg.run_name})' if experiment == 'finetune' else ''
print(f'running {experiment}{tag}  from {CONFIG.name}')
importlib.import_module(EXPERIMENTS[experiment]).run(cfg)

### 3b · Summarize the input-level interventions
Run after **both** intervention configs above. Reads their CSVs and writes the mean change in macro F1 per condition with the paired Wilcoxon tests behind it. `eligibility='heldout'` is the one to report — it excludes any annotated image that was in a given run's training split.

In [ ]:
# Paper-facing numbers for the two input-level interventions (CSV-only; no GPU).
from scripts.compute_intervention_stats import run as run_intervention_stats

run_intervention_stats(paths.results_dir, eligibility='heldout')
# -> results/background_intervention_stats.csv
# -> results/frequency_intervention_stats.csv

## 4 · Standard analysis — needs only `eval_results.csv`
Aggregates, figures, the significance tests, and the decomposition ordering-robustness check. These read CSVs only, so they run without a GPU.

In [ ]:
import sys, runpy
for script in ("aggregate_results", "make_figures", "compute_significance", "compute_decomposition_robustness"):
    sys.argv = [script, "--results-dir", f"{DRIVE}/results"]
    runpy.run_path(str(CODE / "scripts" / f"{script}.py"), run_name="__main__")

## 5 · Checkpoint-based analysis — needs the trained checkpoints
ECE bin-count robustness, class-vs-dataset feature silhouettes (UMAP), the few-shot adaptation curve, and the within/cross confusion matrices. Run after training + evaluation. **No retraining** — they load existing checkpoints and only do inference + analysis. Feature geometry needs `umap-learn`.

In [9]:
# Checkpoint-based analysis: loads existing checkpoints (no retraining).
# Default = full 3x3 split x init seed grid (matches the committed result CSVs).
# Pass quick=True for the faster thesis subset (fixed split seed x init seeds).
!pip install -q umap-learn
from src.config import load_config
from scripts.compute_ece_robustness import run as run_ece
from scripts.compute_feature_geometry import run as run_fg
from scripts.compute_few_shot import run as run_fs
from scripts.compute_coral import run as run_coral
from scripts.compute_confusions import run as run_confusions

cfg = load_config(paths=paths)
from dataclasses import replace
cfg = replace(cfg, training=replace(cfg.training, num_workers=2))  # T4/Colab has ~2 CPUs; the default 8 oversubscribes
#run_ece(cfg)               # -> results/ece_robustness.csv     (quick=True for thesis subset)
#run_fg(cfg, per_class=25)  # -> results/feature_geometry.csv   (quick=True for thesis subset)
#run_fs(cfg)                # -> results/few_shot.csv           (quick=True for thesis subset)
#run_coral(cfg)             # -> results/coral.csv              (zero-label DA baseline)
run_confusions(cfg)        # -> results/confusions.csv         (within + cross 3x3 matrices)

## 6 · Bundle artifacts to Drive

Checkpoints already persist under `MyDrive/<DRIVE_ROOT>/checkpoints/`. This zips them (plus any logs/results) for easy download; evaluation is then run locally against these checkpoints.

In [ ]:
colab.bundle_artifacts(paths, DRIVE)